In [4]:
# ee_batch_export_ndvi_nbar_drive.py
# Batch export Max NDVI (DIY NBAR) to Google Drive by longitude tiles.
import time
import math
import ee

ee.Initialize()

# ---------- Config ----------
START_YEAR    = 2001
END_YEAR      = 2002
LON_STEP_DEG  = 1.0           # if memory/limits persist, try 0.5
TARGET_CRS    = 'EPSG:3031'   # Antarctic Polar Stereographic
TARGET_SCALE  = 100           # 100 m upsample at export
DRIVE_FOLDER  = 'Antarctic_NDVI_DIY_NBAR'  # Drive folder
MAX_ACTIVE    = 80            # throttle active tasks (Drive queues can be picky)

# MODIS constants
RED = 'sur_refl_b01'
NIR = 'sur_refl_b02'
SR_SCALE = 1e-4
A1_SCALE = 1e-3               # set to 1.0 if you confirm GEE already applied scaling
SZA_TARGET_DEG = 45
USE_ANGLE_SCALE_001 = True

# Antarctica bbox (avoid exact -90 deg)
antarctica = ee.Geometry.Polygon([[[-180, -60], [180, -60], [180, -89.9], [-180, -89.9]]])

def season_start(y): return ee.Date.fromYMD(y, 9, 23)
def season_end(y):   return ee.Date.fromYMD(y+1, 3, 21)

# --------------- Helpers ---------------
def to_rad(x): return x.multiply(math.pi/180.0)

def rel_azimuth(saa, vaa):
    r360 = ee.Image.constant(360)
    raa = saa.subtract(vaa).abs()
    raa = raa.min(r360.subtract(raa))
    return to_rad(raa)

def Kvol(SZA, VZA, RAA):
    sz, vz, raa = to_rad(SZA), to_rad(VZA), RAA
    cos_sz, cos_vz = sz.cos(), vz.cos()
    sin_sz, sin_vz = sz.sin(), vz.sin()
    cos_xi = cos_sz.multiply(cos_vz).add(sin_sz.multiply(sin_vz).multiply(raa.cos()))
    xi = cos_xi.acos()
    num = xi.multiply(xi.cos()).multiply(-1).add(xi.sin())
    den = cos_sz.add(cos_vz)
    return num.divide(den).subtract(ee.Image.constant(math.pi).divide(4))

def Kgeo(SZA, VZA, RAA):
    sz, vz, raa = to_rad(SZA), to_rad(VZA), RAA
    cos_sz, cos_vz = sz.cos(), vz.cos()
    sin_sz, sin_vz = sz.sin(), vz.sin()
    tan_sz = sin_sz.divide(cos_sz)
    tan_vz = sin_vz.divide(cos_vz)
    D = tan_sz.multiply(tan_vz).add(raa.cos())
    denom = tan_sz.pow(2).add(1).sqrt().multiply(tan_vz.pow(2).add(1).sqrt())
    cos_t = D.divide(denom)
    t = cos_t.acos()
    O = t.subtract(t.sin().multiply(t.cos())).multiply(1.0/math.pi).multiply(
        cos_sz.add(1).multiply(cos_vz.add(1)))
    return O.subtract(1).multiply(2).divide(cos_sz.add(cos_vz))

def with_day_key(img):
    d = ee.Date(img.get('system:time_start')).format('YYYYMMdd')
    return img.set('dayKey', d)

def a1_select(img):
    bands = [
      'BRDF_Albedo_Parameters_Band1_iso','BRDF_Albedo_Parameters_Band1_vol','BRDF_Albedo_Parameters_Band1_geo',
      'BRDF_Albedo_Parameters_Band2_iso','BRDF_Albedo_Parameters_Band2_vol','BRDF_Albedo_Parameters_Band2_geo'
    ]
    return img.select(bands)

def mask_mod09(img):
    # keep only needed bands early to reduce memory
    img = img.select([RED, NIR, 'state_1km','QC_500m','SolarZenith','SolarAzimuth','SensorZenith','SensorAzimuth'])
    qa = img.select('state_1km')
    cloudState = qa.bitwiseAnd(3)                 # bits 0-1
    cirrus     = qa.rightShift(10).bitwiseAnd(3)  # bits 10-11
    clear1k = cloudState.eq(0).And(cirrus.eq(0))

    modland = img.select('QC_500m').bitwiseAnd(3) # bits 0-1
    clear500 = modland.lte(1)

    refl = img.select([RED, NIR]).multiply(SR_SCALE)
    valid = refl.reduce(ee.Reducer.min()).gt(-0.2).And(refl.reduce(ee.Reducer.max()).lt(1.6))
    return img.updateMask(clear1k).updateMask(clear500).updateMask(valid)

def brdf_correct_to_ndvi(modImg, a1):
    SZA = modImg.select('SolarZenith')
    SAA = modImg.select('SolarAzimuth')
    VZA = modImg.select('SensorZenith')
    VAA = modImg.select('SensorAzimuth')
    if USE_ANGLE_SCALE_001:
        SZA = SZA.multiply(0.01); SAA = SAA.multiply(0.01)
        VZA = VZA.multiply(0.01); VAA = VAA.multiply(0.01)
    RAA = rel_azimuth(SAA, VAA)

    SZA_t = ee.Image.constant(SZA_TARGET_DEG)
    VZA_t = ee.Image.constant(0)
    RAA_t = ee.Image.constant(0)

    Kvol_obs = Kvol(SZA, VZA, RAA)
    Kgeo_obs = Kgeo(SZA, VZA, RAA)
    Kvol_tgt = Kvol(SZA_t, VZA_t, RAA_t)
    Kgeo_tgt = Kgeo(SZA_t, VZA_t, RAA_t)

    f1_iso = a1.select('BRDF_Albedo_Parameters_Band1_iso').multiply(A1_SCALE)
    f1_vol = a1.select('BRDF_Albedo_Parameters_Band1_vol').multiply(A1_SCALE)
    f1_geo = a1.select('BRDF_Albedo_Parameters_Band1_geo').multiply(A1_SCALE)
    f2_iso = a1.select('BRDF_Albedo_Parameters_Band2_iso').multiply(A1_SCALE)
    f2_vol = a1.select('BRDF_Albedo_Parameters_Band2_vol').multiply(A1_SCALE)
    f2_geo = a1.select('BRDF_Albedo_Parameters_Band2_geo').multiply(A1_SCALE)

    R1_obs_m = f1_iso.add(f1_vol.multiply(Kvol_obs)).add(f1_geo.multiply(Kgeo_obs))
    R2_obs_m = f2_iso.add(f2_vol.multiply(Kvol_obs)).add(f2_geo.multiply(Kgeo_obs))
    R1_tgt_m = f1_iso.add(f1_vol.multiply(Kvol_tgt)).add(f1_geo.multiply(Kgeo_tgt))
    R2_tgt_m = f2_iso.add(f2_vol.multiply(Kvol_tgt)).add(f2_geo.multiply(Kgeo_tgt))

    eps = ee.Image.constant(1e-6)
    CF1 = R1_tgt_m.divide(R1_obs_m.where(R1_obs_m.abs().lt(eps), eps))
    CF2 = R2_tgt_m.divide(R2_obs_m.where(R2_obs_m.abs().lt(eps), eps))

    red = modImg.select(RED).multiply(SR_SCALE)
    nir = modImg.select(NIR).multiply(SR_SCALE)
    red_nbar = red.multiply(CF1)
    nir_nbar = nir.multiply(CF2)

    ndvi = nir_nbar.subtract(red_nbar).divide(nir_nbar.add(red_nbar)).rename('ndvi_nbar')
    return ndvi.updateMask(ndvi.gt(0).And(ndvi.lt(1)))

def lon_tiles(step_deg=1.0):
    tiles = []
    lon = -180.0
    while lon < 180:
        lon2 = min(lon + step_deg, 180)
        # simple WGS84 rect; already within your Antarctica bbox
        tile = ee.Geometry.Rectangle([lon, -89.9, lon2, -60], proj=None, geodesic=False)
        tiles.append(tile)
        lon = lon2
    return tiles

def max_ndvi_for_tile(year, tile):
    start, end = season_start(year), season_end(year)
    mod = ee.ImageCollection('MODIS/061/MOD09GA') \
            .filterDate(start, end).filterBounds(tile) \
            .map(mask_mod09).map(with_day_key) \
            .map(lambda i: ee.Image(i).clip(tile))
    myd = ee.ImageCollection('MODIS/061/MYD09GA') \
            .filterDate(start, end).filterBounds(tile) \
            .map(mask_mod09).map(with_day_key) \
            .map(lambda i: ee.Image(i).clip(tile))
    srDaily = mod.merge(myd)

    a1 = ee.ImageCollection('MODIS/061/MCD43A1') \
            .filterDate(start, end).filterBounds(tile) \
            .map(a1_select).map(with_day_key) \
            .map(lambda i: ee.Image(i).clip(tile))

    joined = ee.Join.inner().apply(srDaily, a1, ee.Filter.equals(leftField='dayKey', rightField='dayKey'))

    def pair_to_ndvi(f):
        f = ee.Feature(f)
        sr = ee.Image(f.get('primary'))
        a  = ee.Image(f.get('secondary'))
        nd = brdf_correct_to_ndvi(sr, a)
        return nd.copyProperties(sr, ['system:time_start'])

    ndviDaily = ee.ImageCollection(joined.map(pair_to_ndvi))
    has = ee.Number(ndviDaily.size()).gt(0)
    maxNdvi = ee.Image(ee.Algorithms.If(
        has, ndviDaily.max(), ee.Image(0).updateMask(ee.Image(0)).rename('ndvi_nbar')
    ))
    return maxNdvi.clip(tile).toFloat()

def active_tasks():
    cnt = 0
    for t in ee.batch.Task.list():
        st = t.status().get('state')
        if st in ('READY','RUNNING'):
            cnt += 1
    return cnt

def throttle_start(task):
    # simple throttle to avoid Drive queue issues
    while active_tasks() >= MAX_ACTIVE:
        time.sleep(10)
    task.start()

def start_exports():
    tiles = lon_tiles(LON_STEP_DEG)
    for y in range(START_YEAR, END_YEAR+1):
        for idx, tile in enumerate(tiles):
            img = max_ndvi_for_tile(y, tile)
            desc = f'MaxNDVI_DIYNBAR_{y}_EPSG3031_100m_lon{idx:03d}'
            task = ee.batch.Export.image.toDrive(
                image=img,
                description=desc,
                folder=DRIVE_FOLDER,
                region=tile,              # pass the ee.Geometry directly
                crs=TARGET_CRS,           # 'EPSG:3031'
                scale=TARGET_SCALE,       # 100
                maxPixels=1e13,
                fileFormat='GeoTIFF',
                fileDimensions=8192,      # max pixels per output file side; exporter will split automatically
                skipEmptyTiles=True       # don't write fully-empty tiles
            )
            throttle_start(task)
            print('Started:', desc, '| Active:', active_tasks())

if __name__ == '__main__':
    start_exports()
    print('All tasks submitted.')

Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon000 | Active: 1
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon001 | Active: 2
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon002 | Active: 3
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon003 | Active: 4
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon004 | Active: 5
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon005 | Active: 5
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon006 | Active: 4
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon007 | Active: 4
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon008 | Active: 5
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon009 | Active: 4
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon010 | Active: 3
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon011 | Active: 3
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon012 | Active: 4
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon013 | Active: 4
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon014 | Active: 3
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon015 | Ac

Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon130 | Active: 11
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon131 | Active: 11
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon132 | Active: 10
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon133 | Active: 11
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon134 | Active: 11
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon135 | Active: 11
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon136 | Active: 11
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon137 | Active: 11
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon138 | Active: 11
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon139 | Active: 11
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon140 | Active: 10
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon141 | Active: 11
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon142 | Active: 12
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon143 | Active: 9
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon144 | Active: 10
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_10

Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon260 | Active: 23
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon261 | Active: 23
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon262 | Active: 23
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon263 | Active: 23
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon264 | Active: 23
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon265 | Active: 23
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon266 | Active: 22
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon267 | Active: 22
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon268 | Active: 22
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon269 | Active: 22
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon270 | Active: 23
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon271 | Active: 21
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon272 | Active: 21
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon273 | Active: 22
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_100m_lon274 | Active: 20
Started: MaxNDVI_DIYNBAR_2001_EPSG3031_1

Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon029 | Active: 80
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon030 | Active: 80
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon031 | Active: 80
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon032 | Active: 79
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon033 | Active: 80
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon034 | Active: 79
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon035 | Active: 80
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon036 | Active: 79
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon037 | Active: 79
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon038 | Active: 80
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon039 | Active: 80
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon040 | Active: 79
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon041 | Active: 80
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon042 | Active: 79
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon043 | Active: 80
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_1

Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon158 | Active: 75
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon159 | Active: 73
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon160 | Active: 71
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon161 | Active: 72
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon162 | Active: 72
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon163 | Active: 72
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon164 | Active: 72
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon165 | Active: 71
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon166 | Active: 71
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon167 | Active: 71
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon168 | Active: 72
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon169 | Active: 70
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon170 | Active: 70
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon171 | Active: 71
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon172 | Active: 71
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_1

Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon287 | Active: 77
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon288 | Active: 77
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon289 | Active: 77
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon290 | Active: 76
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon291 | Active: 76
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon292 | Active: 77
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon293 | Active: 78
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon294 | Active: 78
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon295 | Active: 78
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon296 | Active: 78
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon297 | Active: 79
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon298 | Active: 79
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon299 | Active: 80
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon300 | Active: 80
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_100m_lon301 | Active: 80
Started: MaxNDVI_DIYNBAR_2002_EPSG3031_1